# QuantumSCC: Annotated Notebook with Master's Thesis Equations

This notebook maps each code block from the QuantumSCC repository to the corresponding equations in Rubén Gordillo Hachuel's Master's Thesis (TFM).

**Main Reference:** Parra-Rodriguez & Egusquiza, *Geometrical description and Faddeev-Jackiw quantization of electrical networks*, Quantum 8, 1466 (2024)

---

## Index

1. [Constants and Units](#1)
2. [Circuit Elements - elements.py](#2)
3. [Kirchhoff Equations](#3)
4. [Symplectic Form and Faddeev-Jackiw](#4)
5. [Classical Hamiltonian](#5)
6. [Harmonic Diagonalization](#6)
7. [Quantization](#7)
8. [What Remains Uncovered](#8)
9. [QPS. New addition](#9)

In [26]:
import numpy as np
from scipy.linalg import null_space
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded")

Libraries loaded


---
<a id='1'></a>
## 1. Physical Constants (units.py)

The magnetic flux quantum is $\Phi_0 = h/(2e) = 2.067833 \times 10^{-15}$ Wb.

The reduced flux quantum is defined as $\phi_0 = \Phi_0/(2\pi)$.

In [27]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHYSICAL CONSTANTS (units.py)
# ═══════════════════════════════════════════════════════════════════════════════

hbar = 1.0545718e-34      # Reduced Planck constant [J·s]
Phi0 = 2.067833e-15       # Magnetic flux quantum Φ₀ = h/(2e) [Wb]
e = 1.6021766e-19         # Electron charge [C]

freq_list = {'Hz': 1.0, 'kHz': 1.0e3, 'MHz': 1.0e6, 'GHz': 1.0e9, 'THz': 1.0e12}
farad_list = {'F': 1, 'mF': 1.0e-3, 'uF': 1.0e-6, 'nF': 1.0e-9, 'pF': 1.0e-12, 'fF': 1.0e-15}
henry_list = {'H': 1.0, 'mH': 1.0e-3, 'uH': 1.0e-6, 'nH': 1.0e-9, 'pH': 1.0e-12, 'fH': 1.0e-15}
_unit_freq = freq_list["GHz"]

---
<a id='2'></a>
## 2. Circuit Elements (elements.py)

### Thesis Equation 10 - Total Energy Function

$$E(\phi, q) = \sum_{c \in C} h_C^c + \sum_{l \in L} h_L^l - \sum_{j \in J} E_{J_j} \cos\left(\frac{2\pi\phi^j}{\Phi_0}\right)$$

Where:
- **Capacitor:** $h_C^c(q^c) = \frac{(q^c)^2}{2C_c}$
- **Inductor:** $h_L^l(\phi^l) = \frac{(\phi^l)^2}{2L_l}$
- **Josephson Junction:** $h_J^j(\phi^j) = -E_{J_j}\cos\left(\frac{2\pi\phi^j}{\Phi_0}\right)$

### Topological Ansatz (Thesis Figure 1)

| Element | Flux $\phi^b$ | Charge $q^b$ |
|----------|----------------|-------------|
| Capacitor | $\in S^1$ (compact) | $\in \mathbb{R}$ (extended) |
| Inductor | $\in \mathbb{R}$ (extended) | $\in S^1$ (compact) |
| Josephson Junction | $\in S^1$ (compact) | $\in \mathbb{R}$ (extended) |

In [28]:
# ═══════════════════════════════════════════════════════════════════════════════
# CAPACITOR CLASS - Eq. (10): h_C(q) = q²/(2C)
#
# Topological Ansatz (Fig. 1): φ ∈ S¹ (compact), q ∈ ℝ (extended)
# ═══════════════════════════════════════════════════════════════════════════════

class Capacitor:
    """
    Capacitor with charging energy E_C = (2e)²/(2C).
    h_C(q) = q²/(2C) = E_C · n² where n = q/(2e)
    """
    def __init__(self, value, unit='GHz'):
        self.cValue = value
        self.unit = unit
        self.type = type(self)

    def value(self):
        if self.unit in farad_list:
            return self.cValue * farad_list[self.unit]
        else:
            E_c = self.cValue * freq_list[self.unit] * hbar
            return (2*e)**2 / (2 * E_c)

    def energy(self):
        if self.unit in freq_list:
            return self.cValue * freq_list[self.unit] / _unit_freq
        else:
            c = self.cValue * farad_list[self.unit]
            return (2*e)**2 / (2 * c) / hbar / _unit_freq

In [29]:
# ═══════════════════════════════════════════════════════════════════════════════
# INDUCTOR CLASS - Eq. (10): h_L(φ) = φ²/(2L)
#
# Topological Ansatz (Fig. 1): φ ∈ ℝ (extended), q ∈ S¹ (compact)
# ═══════════════════════════════════════════════════════════════════════════════

class Inductor:
    """
    Inductor with inductive energy E_L = (Φ₀/2π)²/(2L) = φ₀²/(2L).
    h_L(φ) = φ²/(2L) = E_L · (φ/φ₀)²
    """
    def __init__(self, value, unit='GHz'):
        self.lValue = value
        self.unit = unit
        self.type = type(self)

    def value(self):
        if self.unit in henry_list:
            return self.lValue * henry_list[self.unit]
        else:
            E_l = self.lValue * freq_list[self.unit] * hbar
            return (Phi0/(2*np.pi))**2 / (2 * E_l)

    def energy(self):
        if self.unit in freq_list:
            return self.lValue * freq_list[self.unit] / _unit_freq
        else:
            l = self.lValue * henry_list[self.unit]
            return (Phi0/(2*np.pi))**2 / (2 * l) / hbar / _unit_freq

In [30]:
# ═══════════════════════════════════════════════════════════════════════════════
# JUNCTION CLASS - Eq. (10): h_J(φ) = -E_J cos(2πφ/Φ₀)
#
# Topological Ansatz (Fig. 1): φ ∈ S¹ (compact), q ∈ ℝ (extended)
# REQUIRES capacitor in parallel (algorithm limitation, Sec. V)
# ═══════════════════════════════════════════════════════════════════════════════

class Junction:
    """
    Josephson Junction: h_J(φ) = -E_J cos(2πφ/Φ₀) = -E_J cos(φ̂)
    Requires a parallel capacitor to avoid non-linear constraints.
    """
    def __init__(self, value, unit='GHz', cap=None):
        if cap is None:
            raise ValueError("Every JJ must have a parallel capacitor")
        self.jValue = value
        self.unit = unit
        self.cap = cap
        self.type = type(self)

    def value(self):
        return self.jValue * freq_list[self.unit] / _unit_freq

### 💡 Possible Improvements in elements.py

| Improvement | Description |  |
|--------|-------------|----------|
| **QPS** | Dual of JJ: $h_{QPS}(q) = -E_S \cos(2\pi q / 2e)$ |  |
| **Validation** | Checks for negative/zero values |  |
| **Documentation** | Clarify $\phi_0$ vs $\Phi_0$ |  |

---
<a id='3'></a>
## 3. Kirchhoff Equations (circuit.py - Kirchhoff)

### Equation 1 - Pfaffian System

$$\text{KCL: } \sum_{b \in \mathcal{N}} dq^b = 0 \qquad \text{KVL: } \sum_{b \in \mathcal{P}} d\phi^b = 0$$

### Equation 2 - Matrix Form

$$F \, dR = 0, \quad F = \begin{pmatrix} F_{loop} & 0 \\ 0 & F_{cut} \end{pmatrix}$$

### Equations 3-5 - Kernel

$$K = \begin{pmatrix} K_{loop} & 0 \\ 0 & F_{loop}^T \end{pmatrix} \quad \text{such that } F K = 0$$

### Equation 6 - Change of Variables

$$R = K Z$$

where $Z^T = (\Phi_c^T, \Phi_e^T, Q^T)$

In [31]:
# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS (algebra.py)
# ═══════════════════════════════════════════════════════════════════════════════

def GaussJordan(M):
    """Gauss-Jordan for F_cut → [I|A]. Ref: Thesis Sec. II.A"""
    nrows, ncolumns = M.shape
    M = M.copy().astype(float)
    order = np.arange(ncolumns)
    for i in range(min(nrows, ncolumns)):
        k = np.argmax(np.abs(M[i, i:]))
        if k != 0:
            M[:, [i, i+k]] = M[:, [i+k, i]]
            order[i], order[i+k] = order[i+k], order[i]
        if np.abs(M[i, i]) > 1e-14:
            for j in range(i+1, nrows):
                M[j, :] -= M[i, :] * M[j, i] / M[i, i]
    return M, order

def reverseGaussJordan(M):
    """Backward elimination for diagonal form"""
    M = M.copy()
    for i in range(M.shape[0]):
        if np.abs(M[i, i]) > 1e-14:
            M[i, :] /= M[i, i]
    for i in reversed(range(M.shape[0])):
        for j in range(i):
            M[j, :] -= M[j, i] * M[i, :]
    return M

def remove_zero_rows(M, tol=1e-14):
    return M[np.sum(np.abs(M), axis=1) > tol, :]

def GS_algorithm(M, normal=True, delete_zeros=True, tol=1e-14):
    """Gram-Schmidt. Ref Thesis [18]: Schmidt, Math. Annalen 63 (1907)"""
    M_out = np.zeros_like(M, dtype=float)
    M_out[:, 0] = M[:, 0]
    if normal and np.linalg.norm(M_out[:, 0]) > tol:
        M_out[:, 0] /= np.linalg.norm(M_out[:, 0])
    for i in range(1, M.shape[1]):
        M_out[:, i] = M[:, i]
        for j in range(i):
            if np.linalg.norm(M_out[:, j]) > tol:
                M_out[:, i] -= np.dot(M[:, i], M_out[:, j]) / np.dot(M_out[:, j], M_out[:, j]) * M_out[:, j]
        if normal and np.linalg.norm(M_out[:, i]) > tol:
            M_out[:, i] /= np.linalg.norm(M_out[:, i])
    if delete_zeros:
        M_out = M_out[:, np.linalg.norm(M_out, axis=0) > tol]
    return M_out

In [32]:
# ═══════════════════════════════════════════════════════════════════════════════
# KIRCHHOFF CONSTRUCTION (Eq. 1-6 of Thesis)
# ═══════════════════════════════════════════════════════════════════════════════

def build_Kirchhoff(elements, no_nodes, no_JJ, no_Capacitors, no_Inductors):
    """
    Eq. (2): F dR = 0, F = [[F_loop, 0], [0, F_cut]]
    Eq. (5): K = [[K_loop, 0], [0, F_loop^T]] with FK = 0
    Eq. (6): R = KZ
    """
    no_elements = len(elements)

    # STEP 1: F_cut from KCL (Eq. 1)
    Fcut = np.zeros((no_nodes, no_elements))
    for n, (orig, dest, _) in enumerate(elements):
        Fcut[orig, n] = -1
        Fcut[dest, n] = +1
    Fcut, order = GaussJordan(Fcut)
    Fcut = reverseGaussJordan(remove_zero_rows(Fcut))

    # STEP 2: F_loop from F_loop @ F_cut^T = 0
    n = len(Fcut)
    A = Fcut[:, n:]
    Floop = np.hstack((-A.T, np.eye(A.shape[1])))
    Fcut = Fcut[:, np.argsort(order)]
    Floop = Floop[:, np.argsort(order)]

    # STEP 3: Complete F (Eq. 2)
    F = np.block([[Floop, np.zeros((Floop.shape[0], Fcut.shape[1]))],
                  [np.zeros((Fcut.shape[0], Floop.shape[1])), Fcut]])

    # STEP 4: Kernel K (Eq. 3-5)
    no_compact = no_JJ + no_Capacitors
    no_extended = no_Inductors
    Floop_S = Floop[:, :no_compact]
    Kloop_S = null_space(Floop_S)
    if Kloop_S.shape[1] > 0:
        Kloop_S = np.vstack((Kloop_S, np.zeros((no_extended, Kloop_S.shape[1]))))
    no_reduced = Kloop_S.shape[1] if Kloop_S.shape[1] > 0 else 0
    Kloop_aux = Fcut.T
    Kloop = Kloop_aux if no_reduced == 0 else GS_algorithm(np.hstack([Kloop_S, Kloop_aux]))
    Kcut = Floop.T
    K = np.block([[Kloop, np.zeros((Kloop.shape[0], Kcut.shape[1]))],
                  [np.zeros((Kcut.shape[0], Kloop.shape[1])), Kcut]])

    assert np.allclose(F @ K, 0), "Error: FK ≠ 0"
    return Fcut, Floop, F, K, no_reduced

# EXAMPLE
C = Capacitor(0.1, 'nF'); L = Inductor(1, 'nH')
LC = [[0,1,C], [0,1,L]]
Fcut, Floop, F, K, nc = build_Kirchhoff(LC, 2, 0, 1, 1)
print(f"F_cut:\n{Fcut}\nF_loop:\n{Floop}\nK:\n{K}")

F_cut:
[[1. 1.]]
F_loop:
[[-1.  1.]]
K:
[[ 1.  0.]
 [ 1.  0.]
 [ 0. -1.]
 [ 0.  1.]]


---
<a id='4'></a>
## 4. Symplectic Form (omega_function)

### Equation 8 - Two-form
$$\omega_{2B} = \frac{1}{2} dR^T \wedge \Omega_{2B} \, dR$$

### Equation 14 - Faddeev-Jackiw Reduction
$$V^T \Omega V = \begin{pmatrix} J & 0 \\ 0 & 0 \end{pmatrix}, \quad J = \begin{pmatrix} 0 & I \\ -I & 0 \end{pmatrix}$$

In [33]:
# ═══════════════════════════════════════════════════════════════════════════════
# TWO-FORM Ω_{2B} (Eq. 8)
# Signs according to ansatz: JJ/Inductor: +½, Capacitor: -½
# ═══════════════════════════════════════════════════════════════════════════════

def build_omega_2B(elements, no_elements):
    """Constructs antisymmetric Ω_{2B} according to Eq. (8)"""
    omega = np.zeros((2*no_elements, 2*no_elements))
    for i, (_, _, elem) in enumerate(elements):
        if isinstance(elem, Junction):
            omega[i, i+no_elements] = 0.5
            omega[i+no_elements, i] = -0.5
        elif isinstance(elem, Capacitor):
            omega[i, i+no_elements] = -0.5
            omega[i+no_elements, i] = 0.5
        elif isinstance(elem, Inductor):
            omega[i, i+no_elements] = 0.5
            omega[i+no_elements, i] = -0.5
    return omega

---
<a id='5'></a>
## 5. Classical Hamiltonian

### Equation 11 - Quadratic Energy
$$\sum h_C + \sum h_L = \frac{1}{2} R^T E_{2B} R$$

### Equation 18 - Elimination of Gauge Variables
$$H = \tilde{H}_{\xi\xi} - \tilde{H}_{\xi w} \tilde{H}_{ww}^{-1} \tilde{H}_{w\xi}$$

### Equation 19 - Final Hamiltonian
$$H(\xi) = \frac{1}{2}\xi^T H \xi - \sum_{j} E_{J_j} \cos(\hat{\phi}_j)$$

In [34]:
# ═══════════════════════════════════════════════════════════════════════════════
# PSEUDOINVERSE - Ref Thesis [19]: Penrose 1955
# ═══════════════════════════════════════════════════════════════════════════════

def pseudo_inv(M, tol=1e-15):
    U, S, Vt = np.linalg.svd(M)
    S_inv = np.zeros((Vt.shape[0], U.shape[1]))
    for i in range(len(S)):
        if np.abs(S[i]) > tol: S_inv[i,i] = 1/S[i]
    return Vt.T @ S_inv @ U.T

# ═══════════════════════════════════════════════════════════════════════════════
# CLASSICAL HAMILTONIAN (Eq. 11, 17-19)
# ═══════════════════════════════════════════════════════════════════════════════

def build_hamiltonian(elements, K, V, no_elem, no_indep):
    """Eq. (18): H = H̃_ξξ - H̃_ξw H̃_ww⁻¹ H̃_wξ"""
    E_2B = np.zeros((2*no_elem, 2*no_elem))
    for i, (_, _, elem) in enumerate(elements):
        if isinstance(elem, Inductor): E_2B[i,i] = 2*elem.energy()
        elif isinstance(elem, Capacitor): E_2B[i+no_elem, i+no_elem] = 2*elem.energy()
    E_sym = V.T @ K.T @ E_2B @ K @ V
    if E_sym.shape[0] == no_indep:
        return E_sym
    n = no_indep
    return E_sym[:n,:n] - E_sym[:n,n:] @ pseudo_inv(E_sym[n:,n:]) @ E_sym[n:,:n]

---
<a id='6'></a>
## 6. Harmonic Diagonalization (Section III)

### Equations 22-24 - Symplectic Normalization
$$\alpha(g, \bar{g}) = g^T J \bar{g}, \quad e = \frac{g}{\sqrt{\sigma\alpha}}$$

$$T = [T_+ | T_-], \quad t = \sqrt{2}\text{Re}\{e\}, \quad s = \sqrt{2}\text{Im}\{e\}$$

### Equations 25-26 - Second Quantization
$$G = \frac{1}{\sqrt{2}}\begin{pmatrix} I & I \\ -iI & iI \end{pmatrix}$$

$$\hat{H}_e = \sum_j \hbar\omega_j \hat{a}_j^\dagger \hat{a}_j$$

When we defined the circuit the quantization is already done


In [35]:
# ═══════════════════════════════════════════════════════════════════════════════
# SYMPLECTIC DIAGONALIZATION (Eq. 22-26)
# Ref Thesis [14]: Kustura et al., PRA 99, 022130 (2019)
# ═══════════════════════════════════════════════════════════════════════════════

def symplectic_diag(H, tol=1e-14):
    dim = H.shape[0]; n = dim//2
    J = np.block([[np.zeros((n,n)), np.eye(n)], [-np.eye(n), np.zeros((n,n))]])
    eigvals, eigvecs = np.linalg.eig(J @ H)
    idx = np.argsort(eigvals.imag)
    eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]

    pos_vecs = [eigvecs[:,i] for i, ev in enumerate(eigvals) if np.abs(ev.real)<tol and ev.imag>tol]
    freqs = [ev.imag for ev in eigvals if np.abs(ev.real)<tol and ev.imag>tol]

    norm_vecs = []
    for g in pos_vecs:
        alpha = g @ J @ np.conj(g)
        sigma = 1j * np.sign(alpha.imag) if np.abs(alpha.imag)>tol else 1j
        norm_vecs.append(g / np.sqrt(np.abs(sigma*alpha)) if np.abs(sigma*alpha)>tol else g)

    T = np.hstack([np.sqrt(2)*np.real(e).reshape(-1,1) for e in norm_vecs] +
                  [np.sqrt(2)*np.imag(e).reshape(-1,1) for e in norm_vecs])
    I = np.eye(n)
    G = (1/np.sqrt(2)) * np.block([[I, I], [-1j*I, 1j*I]])
    H_diag = np.real(np.conj(G.T) @ T.T @ H @ T @ G)
    return H_diag, T, G, np.array(freqs)

# # LC EXAMPLE
# phi0 = Phi0/(2*np.pi); L_val, C_val = 1e-9, 0.1e-9
# E_L = phi0**2/(2*L_val)/hbar/1e9; E_C = (2*e)**2/(2*C_val)/hbar/1e9
# H_LC = np.array([[2*E_L,0],[0,2*E_C]])
# H_d, T, G, f = symplectic_diag(H_LC)
# print(f"LC Frequency: {f[0]:.3f} GHz (Theoretical: {1/np.sqrt(L_val*C_val)/2/np.pi/1e9:.3f} GHz)")

---
<a id='7'></a>
## 7. Quantization (Section II.C)

### Equation 20 - Quantum Hamiltonian
$$\hat{H} = \frac{1}{2}\hat{\xi}^T H \hat{\xi} - \sum_j E_{J_j}\cos(\hat{\phi}_j)$$

**Extended Variables:** $[\hat{\Phi}, \hat{Q}] = i\hbar$

**Compact Variables:** $n = Q/(2e)$, $\phi = 2\pi\Phi/\Phi_0$, $[\hat{n}, e^{i\hat{\phi}}] = e^{i\hat{\phi}}$

---
<a id='8'></a>
## 8. What Remains Uncovered (Sec. VI of Thesis)

In [36]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                    SUMMARY: STATUS vs THESIS                             ║
╠══════════════════════════════════════════════════════════════════════════╣
║    IMPLEMENTED                                                           ║
║ • Kirchhoff matrices F_cut, F_loop, kernel K (Eq. 1-6)                   ║
║ • Two-form ω_2B and symplectic transformation (Eq. 8, 14)                ║
║ • Classical Hamiltonian with Schur (Eq. 17-19)                           ║
║ • Harmonic diagonalization (Eq. 22-26)                                   ║
║ • Examples: LC, coupled, fluxonium, singular                             ║
╠══════════════════════════════════════════════════════════════════════════╣
║    NOT IMPLEMENTED (Future Work Sec. VI)                                 ║
║ • Diagonalization of NON-LINEAR subspace (Fock basis)                    ║
║ • QPS (Quantum Phase Slip) - dual of JJ                                  ║
║  • External fluxes Φ_ext in superconducting loops                        ║
╚══════════════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════════════╗
║                    SUMMARY: STATUS vs THESIS                             ║
╠══════════════════════════════════════════════════════════════════════════╣
║    IMPLEMENTED                                                           ║
║ • Kirchhoff matrices F_cut, F_loop, kernel K (Eq. 1-6)                   ║
║ • Two-form ω_2B and symplectic transformation (Eq. 8, 14)                ║
║ • Classical Hamiltonian with Schur (Eq. 17-19)                           ║
║ • Harmonic diagonalization (Eq. 22-26)                                   ║
║ • Examples: LC, coupled, fluxonium, singular                             ║
╠══════════════════════════════════════════════════════════════════════════╣
║    NOT IMPLEMENTED (Future Work Sec. VI)                                 ║
║ • Diagonalization of NON-LINEAR subspace (Fock basis)                    ║
║ • QPS (Quantum Phase Slip) - dual of JJ                                  

### Code ↔ Equations Correspondence Table

| File | Method | Equations | Section |
|---------|--------|------------|--------|
| elements.py | Capacitor/Inductor/Junction.energy() | Eq. 10 | II.B |
| circuit.py | Kirchhoff() | Eq. 1-6 | II.A |
| circuit.py | omega_function() | Eq. 7-9, 14-16 | II.B |
| circuit.py | classical_hamiltonian_function() | Eq. 10-13, 17-19 | II.B |
| circuit.py | extended_hamiltonian_quantization() | Eq. 20-26 | III |
| algebra.py | GaussJordan() | Sec. II.A | II.A |
| algebra.py | pseudo_inv() | Ref. [19] | II.B |
| algebra.py | symplectic_transformation() | Eq. 22-24 | III |

<a id='9'></a>
## 9 Addition_ QPS beta version not implemented yet

In [37]:
class QuantumPhaseSlip:
    """
    Class that contains the Quantum Phase Slip (QPS) properties.
    This element is the electromagnetic dual of the Josephson Junction.

    Topological Ansatz:
    - Flux: Extended (R)
    - Charge: Compact (S1)

    Parameters
    -----------
    value: float
        The value of the Phase Slip energy (Es).
    unit: str, optional
        The unit of the input value. Typically "GHz", "MHz", etc.
        If None, defaults to the global JJ unit (usually "GHz").
    ind: Inductor, optional
        A series inductor associated with the QPS.
        (Analogous to the parallel capacitor required for Josephson Junctions
        to avoid singular transformations).
    """

    def __init__(
        self,
        value: float,
        unit: Optional[str] = None,
        ind: Optional[Any] = None,
    ) -> None:

        if unit not in unt.freq_list and unit is not None:
            error = (
                "The input unit for the QPS is not correct. "
                "Look at the documentation for the correct input format."
            )
            raise ValueError(error)

        self.sValue = value
        self.type = type(self)

        # Similar to how JJs require a parallel cap, a QPS theoretically
        # requires a series inductor for regularization in this formalism.
        self.ind = ind

        if unit is None:
            self.unit = unt.get_unit_JJ() # We reuse the JJ default unit
        else:
            self.unit = unit

    def value(self) -> float:
        """
        Return the energy value Es in the system's frequency unit (e.g., GHz).
        """
        sMean = self.sValue * unt.freq_list[self.unit] / unt.get_unit_freq()
        return sMean

    def energy(self) -> float:
        """
        Returns the energy amplitude for the Hamiltonian term:
        H_QPS = -Es * cos(2*pi*Q / 2e)
        """
        return self.value()

NameError: name 'Optional' is not defined

Need to modify the logic on circuit.py to apply topology constrains and energy functions build

The Circuit class (specifically the algorithms in __init__, Kirchhoff, and classical_hamiltonian_function) assumes only JJs produce non-linearities and that only JJs/Capacitors have compact variables.

A. Initialization (_ _ init _ _)

*   Current logic: Separates elements into no_JJ, no_Capacitors, no_Inductors.
*   Modification: Add no_QPS.
*   Important: When building complete_elements, QPS elements should be treated similarly to Inductors regarding their position (since they are inductive branches), or appended at the end.

B. Kirchoff Matrices

*  Current logic: no_initial_compact_flux_variables = self.no_JJ + self.no_Capacitors.
*   QPS Physics: A QPS has Extended Flux ($\phi \in \mathbb{R}$) and Compact Charge ($q \in S^1$)3
*   Add QPS to no_initial_extended_flux_variables

C. Symplectic form

* Current logic: * JJ/Inductor: $+0.5$ / $-0.5$ (Flux $\to$ Charge).
Capacitor: $-0.5$ / $+0.5$.

* Modification: Since the QPS is an inductive element (it stores energy in the "charge-like" cosine potential, but topologically it acts on the flux branch), it should follow the Inductor sign convention in omega_2B


D. Hamiltonian Construction (classical_hamiltonian_function)

* Current logic: $H = \dots - \sum E_J \cos(\text{vector}_{JJ}^T \cdot R)$. This effectively calculates $\cos(\phi)$.

* Modification: Create a new vector vector_QPS. While vector_JJ selects the Flux variable (indices $0$ to $B$), vector_QPS must select the Charge variable (indices $B$ to $2B$) corresponding to the QPS branch.The final Hamiltonian must include a new term: $$H_{total} = H_{quadratic} - \sum E_J \cos(\varphi) - \sum E_S \cos(2\pi n)$$


